# 00 Part 2 — Download Product Images (per BRAND_BUCKET)

**Input :** `data/{BRAND_BUCKET}/amzn_shoes_monthly_diffs_ffill_fixed_splits/all_products_clean.parquet`
**Output :** `data/images/{ASIN}.jpg` (shared across buckets — no duplication)

Set `BRAND_BUCKET` in cell ② and run. Already-downloaded ASINs are skipped.

## ① Setup

In [8]:
!pip install pyarrow pandas requests Pillow tqdm -q
print('✅ Done')

✅ Done


## ② Config

In [9]:
import os
from pathlib import Path

def _find_project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'data_preparation').exists() and (p / 'code').exists():
            return p
    raise RuntimeError(
        'Cannot find project root.\n'
        'Expected a directory containing both "data_preparation/" and "code/".\n'
        f'Searched from: {Path.cwd()}'
    )

# ── BRAND BUCKET — change this line for each run ───────────────────────
# Valid: women_top1_dream_pairs, women_top2_idifu, women_other,
#        men_top1_bruno_marc,    men_top2_nike,    men_other
BRAND_BUCKET = 'men_adidas'

PROJECT_ROOT = _find_project_root()
DATA_ROOT    = PROJECT_ROOT / 'data' / BRAND_BUCKET
DATA_DIR     = DATA_ROOT / 'amzn_shoes_monthly_diffs_ffill_fixed_splits'
IMG_DIR      = PROJECT_ROOT / 'data' / 'images'    # shared (ASIN-unique)
PRED_DIR     = DATA_ROOT / 'predictions'
KEY_DIR      = DATA_ROOT / 'keys'
CLEAN_FILE   = DATA_DIR / 'all_products_clean.parquet'

MAX_RETRIES = 3
TIMEOUT_SEC = 10
SLEEP_SEC   = 0.2

assert CLEAN_FILE.exists(), f'Missing {CLEAN_FILE} — run 00_part1 with this BRAND_BUCKET first'

os.makedirs(IMG_DIR, exist_ok=True)

print(f'BRAND_BUCKET : {BRAND_BUCKET}')
print(f'Input clean  : {CLEAN_FILE}')
print(f'Images out   : {IMG_DIR}  (shared across all buckets)')

BRAND_BUCKET : men_adidas
Input clean  : /Users/yvony/PycharmProjects/Amazon_demand_analysis_seperate/data/men_adidas/amzn_shoes_monthly_diffs_ffill_fixed_splits/all_products_clean.parquet
Images out   : /Users/yvony/PycharmProjects/Amazon_demand_analysis_seperate/data/images  (shared across all buckets)


## ③ Load Clean ASINs and Image URLs from Saved Parquets

In [10]:
import pandas as pd

# Load all clean ASINs — split hasn't happened yet at this stage
df_all = pd.read_parquet(CLEAN_FILE, columns=['ASIN', 'image'])

# One row per ASIN — image URL is static per product
asin_urls = (
    df_all[['ASIN', 'image']]
    .drop_duplicates('ASIN')
    .dropna(subset=['image'])
    .reset_index(drop=True)
)

print(f'Total unique ASINs : {asin_urls["ASIN"].nunique():,}')
print(f'ASINs with image URL : {len(asin_urls):,}')
print(f'ASINs missing image  : {df_all["ASIN"].nunique() - len(asin_urls):,}')
print()
print('Sample URLs:')
for _, row in asin_urls.head(3).iterrows():
    print(f'  {row["ASIN"]} → {row["image"]}')

Total unique ASINs : 65
ASINs with image URL : 65
ASINs missing image  : 0

Sample URLs:
  B01BPL5RBO → https://images-na.ssl-images-amazon.com/images/I/61ou+S162sL.jpg
  B07NQLK865 → https://images-na.ssl-images-amazon.com/images/I/61oAztaWNmL.jpg
  B08CYCFMBQ → https://images-na.ssl-images-amazon.com/images/I/81cuTtP+n6L.jpg


## ④ Check Already Downloaded Images

In [11]:
already_done = set()
if os.path.exists(IMG_DIR):
    for fname in os.listdir(IMG_DIR):
        if fname.endswith('.jpg'):
            already_done.add(fname.replace('.jpg', ''))

to_download = asin_urls[~asin_urls['ASIN'].isin(already_done)].reset_index(drop=True)

print(f'Already downloaded : {len(already_done):,} images')
print(f'To download        : {len(to_download):,} images')
print(f'Target total       : {len(asin_urls):,} images')

Already downloaded : 54 images
To download        : 65 images
Target total       : 65 images


## ⑤ Download Images

In [12]:
import requests
import time
from PIL import Image
from io import BytesIO
from tqdm.auto import tqdm

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    )
}

success  = []
failed   = []
skipped  = list(already_done)

for _, row in tqdm(to_download.iterrows(), total=len(to_download), desc='Downloading'):
    asin      = row['ASIN']
    url       = row['image']
    save_path = IMG_DIR / f'{asin}.jpg'

    downloaded = False
    for attempt in range(MAX_RETRIES):
        try:
            response = requests.get(url, headers=HEADERS, timeout=TIMEOUT_SEC)
            if response.status_code == 200:
                img = Image.open(BytesIO(response.content))
                img = img.convert('RGB')
                img.save(save_path, 'JPEG')
                success.append(asin)
                downloaded = True
                break
            else:
                time.sleep(1)
        except Exception:
            time.sleep(1)

    if not downloaded:
        failed.append({'ASIN': asin, 'url': url})

    time.sleep(SLEEP_SEC)

print(f'\n✅ Downloaded : {len(success):,}')
print(f'⏭  Skipped    : {len(skipped):,}  (already existed)')
print(f'❌ Failed     : {len(failed):,}')

Downloading: 100%|██████████| 65/65 [00:26<00:00,  2.46it/s]


✅ Downloaded : 65
⏭  Skipped    : 54  (already existed)
❌ Failed     : 0


## ⑥ Failed Downloads — Review and Retry if Needed

In [13]:
if len(failed) > 0:
    print(f'Failed ASINs ({len(failed)} total):')
    for item in failed:
        print(f'  {item["ASIN"]} → {item["url"]}')
    print()
    pct_failed = len(failed) / len(asin_urls) * 100
    if pct_failed < 5:
        print(f'Failure rate: {pct_failed:.1f}% — acceptable. Re-run this notebook to retry.')
    else:
        print(f'Failure rate: {pct_failed:.1f}% — higher than expected. Check internet connection.')
else:
    print('✅ All images downloaded successfully — no failures!')

✅ All images downloaded successfully — no failures!


## ⑦ Final Summary

In [14]:
from PIL import Image as PILImage

final_count = len([f for f in os.listdir(IMG_DIR) if f.endswith('.jpg')])
total_size  = sum(
    os.path.getsize(IMG_DIR / f)
    for f in os.listdir(IMG_DIR)
    if f.endswith('.jpg')
) / 1e6

print('=' * 55)
print('IMAGE DOWNLOAD SUMMARY')
print('=' * 55)
print(f'Images on disk  : {final_count:,}')
print(f'Target          : {len(asin_urls):,}')
print(f'Coverage        : {final_count/len(asin_urls)*100:.1f}%')
print(f'Total size      : {total_size:.1f} MB')
print(f'Location        : {IMG_DIR}')
print()

# Verify a few image dimensions
sample_files = [f for f in os.listdir(IMG_DIR) if f.endswith('.jpg')][:5]
print('Sample image sizes:')
for fname in sample_files:
    img = PILImage.open(IMG_DIR / fname)
    print(f'  {fname.replace(".jpg","")} : {img.size[0]} x {img.size[1]} px')

print()

# Cross-check: confirm all clean ASINs have an image
clean_asins    = set(asin_urls['ASIN'].unique())
images_on_disk = set(f.replace('.jpg', '') for f in os.listdir(IMG_DIR) if f.endswith('.jpg'))
missing_images = clean_asins - images_on_disk

if len(missing_images) == 0:
    print(f'✅ All {len(clean_asins):,} clean ASINs have an image on disk.')
else:
    print(f'⚠️  {len(missing_images)} clean ASINs still missing images:')
    for asin in sorted(missing_images)[:20]:
        print(f'  {asin}')
    if len(missing_images) > 20:
        print(f'  ... and {len(missing_images)-20} more')
    print('Re-run from Cell ⑤ to retry.')

print()
if final_count >= len(asin_urls) * 0.95:
    print('✅ Ready for 00_part2b_embed_cluster_split.ipynb (GPU required)')
else:
    print('⚠️  Coverage below 95% — re-run Cell ⑤ to retry failed downloads.')

IMAGE DOWNLOAD SUMMARY
Images on disk  : 119
Target          : 65
Coverage        : 183.1%
Total size      : 18.3 MB
Location        : /Users/yvony/PycharmProjects/Amazon_demand_analysis_seperate/data/images

Sample image sizes:
  B0BZBLDTH3 : 1600 x 1067 px
  B0CZHNJPS2 : 1969 x 2560 px
  B0CTBLFZ6D : 1100 x 1100 px
  B09WML6KHL : 1100 x 723 px
  B0BWTDFGBF : 1969 x 2560 px

✅ All 65 clean ASINs have an image on disk.

✅ Ready for 00_part2b_embed_cluster_split.ipynb (GPU required)
